# Feature Extractor Audit
## Systematic check of all 10 extractors across player profiles

Tests star players, bench players, traded players, rookies, and diacritics players.
Identifies dead features, silent failures, and data gaps.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Set DB credentials
os.environ['DB_USER'] = 'mlb_user'
os.environ['DB_PASSWORD'] = 'mlb_secure_2025'

sys.path.insert(0, '..')
from nba.features.extract_live_features_xl import LiveFeatureExtractorXL

In [ ]:
# Test players
TEST_PLAYERS = [
    ('Nikola Jokic', '2025-02-15', 'POINTS', 'LAL', True, 28.5, 'STAR'),
    ('LeBron James', '2025-01-10', 'POINTS', 'CLE', False, 25.5, 'STAR'),
    ('Scottie Barnes', '2025-03-01', 'REBOUNDS', 'MIA', True, 8.5, 'STAR'),
    ('Max Christie', '2025-02-20', 'POINTS', 'SAC', True, 8.5, 'BENCH'),
    ('Luka Doncic', '2025-03-01', 'POINTS', 'BOS', True, 30.5, 'DIACRITICS'),
    ('Dennis Schroder', '2025-02-10', 'POINTS', 'MIA', False, 15.5, 'DIACRITICS'),
]

ext = LiveFeatureExtractorXL()
results = {}

for name, date, stat, opp, home, line, label in TEST_PLAYERS:
    features = ext.extract_features(name, date, stat, opp, home, line)
    if features:
        results[f'{name} ({label})'] = features
        print(f'{label}: {name} — {len(features)} features extracted')
    else:
        print(f'{label}: {name} — RETURNED NONE')

ext.close()

In [ ]:
# Build comparison DataFrame
df = pd.DataFrame(results).T
print(f'Players: {len(df)}')
print(f'Features: {len(df.columns)}')

# Feature health
always_zero = [c for c in df.columns if (df[c] == 0).all()]
always_same = [c for c in df.columns if df[c].nunique() == 1 and c not in always_zero]
varying = [c for c in df.columns if df[c].nunique() > 1]

print(f'\nHealthy (vary across players): {len(varying)}')
print(f'Always zero: {len(always_zero)}')
print(f'Always same value: {len(always_same)}')

In [ ]:
# Dead features
print('=== ALWAYS ZERO ===')
for c in sorted(always_zero):
    print(f'  {c}')

print(f'\n=== ALWAYS SAME VALUE ===')
for c in sorted(always_same):
    print(f'  {c} = {df[c].iloc[0]}')

In [ ]:
# V4 feature groups
groups = {
    'BP Analytics': [c for c in df.columns if 'bp_analytics' in c or 'dvp_' in c],
    'Game Context': ['game_pace','opp_score_margin_avg','player_minutes_stability',
                     'player_plus_minus_L5','player_usage_proxy','player_scoring_efficiency',
                     'player_blowout_risk','player_minutes_vs_avg'],
    'Temporal': ['is_post_trade_deadline','days_since_trade_deadline','is_post_allstar',
                'season_pct','player_games_with_team','is_new_team','team_tenure_games'],
    'H2H': [c for c in df.columns if c.startswith('h2h_')],
    'Book Features': [c for c in df.columns if 'deviation' in c or 'book' in c],
}

for group_name, cols in groups.items():
    present = [c for c in cols if c in df.columns]
    populated = [c for c in present if df[c].nunique() > 1 or (df[c] != 0).any()]
    print(f'{group_name}: {len(populated)}/{len(present)} populated')

In [ ]:
# Cross-reference with production models
import pickle
v3_feat = set(pickle.load(open('nba/models/saved_xl/points_v3_features.pkl', 'rb')))
xl_feat = set(pickle.load(open('nba/models/saved_xl/points_xl_features.pkl', 'rb')))
live_feat = set(df.columns)

v3_missing = v3_feat - live_feat - {'expected_diff'}
xl_missing = xl_feat - live_feat - {'expected_diff'}

print(f'V3 model needs {len(v3_feat)} features')
print(f'  Missing from live extraction: {len(v3_missing)}')
for f in sorted(v3_missing):
    print(f'    {f}')

print(f'\nXL model needs {len(xl_feat)} features')
print(f'  Missing from live extraction: {len(xl_missing)}')
for f in sorted(xl_missing):
    print(f'    {f}')